# 🐔 CFarm Temperature Prediction

Train model dự đoán nhiệt độ trong chuồng dựa trên:
- Thời gian (giờ trong ngày, ngày trong tuần)
- Độ ẩm, khí gas (MQ135, MQ137)
- Tuổi đợt nuôi

**Dataset:** `ml_training_data_2026-05-24.csv`

In [ ]:
# Install dependencies
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Ready!')

## 1. Load Data

In [ ]:
# Upload file từ máy tính hoặc mount Google Drive
# Cách 1: Upload trực tiếp
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

# Cách 2: Đọc từ Google Drive (uncomment nếu dùng Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/ml_training_data_2026-05-24.csv')

In [ ]:
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# Kiểm tra missing values
df.isnull().sum()

## 2. Data Cleaning

In [ ]:
# Loại bỏ rows có temperature = 0 hoặc missing
df_clean = df[df['temperature'] > 0].copy()

# Điền missing values cho sensor gas bằng median theo barn
for col in ['mq135', 'mq137']:
    df_clean[col] = df_clean.groupby('barn_id')[col].transform(
        lambda x: x.fillna(x.median())
    )
    # Nếu vẫn còn NaN, fill global median
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Drop rows còn lại có NaN
df_clean = df_clean.dropna()

print(f'Shape sau clean: {df_clean.shape}')
df_clean.isnull().sum()

## 3. Feature Engineering

In [ ]:
# Tạo features bổ sung
df_clean['hour_sin'] = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['hour_cos'] = np.cos(2 * np.pi * df_clean['hour'] / 24)

df_clean['dow_sin'] = np.sin(2 * np.pi * df_clean['day_of_week'] / 7)
df_clean['dow_cos'] = np.cos(2 * np.pi * df_clean['day_of_week'] / 7)

# Độ ẩm tương đối (RH) - thường ảnh hưởng nhiệt độ
# Nếu humidity cao + temp cao = nguy hiểm cho gia cầm
df_clean['heat_index'] = df_clean['temperature'] * df_clean['humidity'] / 100

# Gas ratio - phát hiện ammonia
df_clean['gas_ratio'] = df_clean['mq135'] / (df_clean['mq137'] + 1)

df_clean.head()

## 4. EDA - Khám phá dữ liệu

In [ ]:
# Phân bố nhiệt độ theo giờ trong ngày
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.boxplot(data=df_clean, x='hour', y='temperature', ax=axes[0])
axes[0].set_title('Nhiệt độ theo giờ trong ngày')
axes[0].set_xlabel('Giờ')
axes[0].set_ylabel('Nhiệt độ (°C)')

sns.boxplot(data=df_clean, x='day_of_week', y='temperature', ax=axes[1])
axes[1].set_title('Nhiệt độ theo ngày trong tuần')
axes[1].set_xlabel('Ngày trong tuần (0=Mon, 6=Sun)')
axes[1].set_ylabel('Nhiệt độ (°C)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['temperature', 'humidity', 'mq135', 'mq137', 'hour', 'day_of_week', 'day_age']
plt.figure(figsize=(8, 6))
sns.heatmap(df_clean[numeric_cols].corr(), annot=True, fmt='.2f', cmap='RdYlBu_r', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Nhiệt độ theo tuổi đợt nuôi
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df_clean, x='day_age', y='temperature', hue='barn_id', alpha=0.6)
plt.title('Nhiệt độ theo tuổi đợt nuôi')
plt.xlabel('Ngày tuổi')
plt.ylabel('Nhiệt độ (°C)')
plt.legend(title='Chuồng')
plt.show()

## 5. Prepare Training Data

In [ ]:
# Features và target
feature_cols = [
    'hour', 'day_of_week', 'day_of_month', 'day_age',
    'humidity', 'mq135', 'mq137',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'heat_index', 'gas_ratio'
]

X = df_clean[feature_cols].values
y = df_clean['temperature'].values

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test: {X_test.shape[0]} samples')

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Train Models

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
try:
    import xgboost as xgb
    HAS_XGB = True
except:
    HAS_XGB = False
    print('XGBoost not available, skipping...')

models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
}

if HAS_XGB:
    models['XGBoost'] = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)

results = {}
for name, model in models.items():
    # Use scaled data for linear models, original for tree-based
    if 'Regression' in name or 'Ridge' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'model': model}
    print(f'{name:20s} | MAE: {mae:.2f}°C | RMSE: {rmse:.2f}°C | R²: {r2:.3f}')


## 7. Evaluate Best Model

In [ ]:
# Tìm best model theo R²
best_name = max(results, key=lambda k: results[k]['R2'])
best_result = results[best_name]
best_model = best_result['model']

print(f'🏆 Best Model: {best_name}')
print(f'   MAE: {best_result["MAE"]:.2f}°C')
print(f'   RMSE: {best_result["RMSE"]:.2f}°C')
print(f'   R²: {best_result["R2"]:.3f}')

# Prediction vs Actual
if 'Regression' in best_name or 'Ridge' in best_name:
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_best, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Temperature (°C)')
plt.ylabel('Predicted Temperature (°C)')
plt.title(f'{best_name} - Prediction vs Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Residuals distribution
residuals = y_test - y_pred_best
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(residuals, bins=30, edgecolor='black')
plt.axvline(x=0, color='r', linestyle='--')
plt.xlabel('Residual (°C)')
plt.ylabel('Frequency')
plt.title('Residuals Distribution')

plt.subplot(1, 2, 2)
plt.scatter(y_pred_best, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted (°C)')
plt.ylabel('Residual (°C)')
plt.title('Residuals vs Predicted')

plt.tight_layout()
plt.show()

## 8. Feature Importance (Tree-based models)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=True)
    
    plt.figure(figsize=(8, 6))
    plt.barh(importance['feature'], importance['importance'])
    plt.xlabel('Importance')
    plt.title(f'{best_name} - Feature Importance')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importance not available for this model type')

## 9. Save Model

In [ ]:
import joblib

# Save model và scaler
joblib.dump(best_model, '/content/cfarm_temp_model.joblib')
joblib.dump(scaler, '/content/cfarm_scaler.joblib')

print('✅ Model saved:')
print('   - /content/cfarm_temp_model.joblib')
print('   - /content/cfarm_scaler.joblib')

# Download
files.download('/content/cfarm_temp_model.joblib')

In [ ]:
# Quick prediction example
def predict_temperature(hour, day_of_week, day_of_month, day_age, humidity, mq135, mq137):
    # Tạo features
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    dow_sin = np.sin(2 * np.pi * day_of_week / 7)
    dow_cos = np.cos(2 * np.pi * day_of_week / 7)
    heat_index = (df_clean['temperature'].mean() * humidity) / 100  # estimate
    gas_ratio = mq135 / (mq137 + 1)
    
    features = np.array([[hour, day_of_week, day_of_month, day_age,
                          humidity, mq135, mq137,
                          hour_sin, hour_cos, dow_sin, dow_cos,
                          heat_index, gas_ratio]])
    
    features_scaled = scaler.transform(features)
    if 'Regression' in best_name or 'Ridge' in best_name:
        pred = best_model.predict(features_scaled)[0]
    else:
        pred = best_model.predict(features)[0]
    
    return pred

# Ví dụ: Dự đoán nhiệt độ lúc 14h, thứ 5, ngày 25, đợt nuôi 30 ngày, độ ẩm 70%
pred = predict_temperature(
    hour=14, day_of_week=3, day_of_month=25, day_age=30,
    humidity=70, mq135=150, mq137=30
)
print(f'Predicted temperature: {pred:.1f}°C')

---

**Next steps:**
1. Train với more data (thêm barns)
2. Thêm features: weather data, feed consumption
3. Fine-tune hyperparameters với Optuna/GridSearch
4. Deploy lên production (CFarm backend)